# 02. Calidad, limpieza y preprocesamiento

Se conserva cada ID estable y cada texto original. Las transformaciones se crean en columnas
nuevas para que el proceso sea auditable.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Busca los CSV sin depender de una ruta absoluta del equipo
def localizar_archivo(nombre):
    candidatos = [
        Path.cwd() / nombre,
        Path.cwd() / "data" / nombre,
        Path.cwd().parent / "data" / nombre,
    ]

    for ruta in candidatos:
        if ruta.exists():
            return ruta
    buscadas = "\n- ".join(str(p) for p in candidatos)
    
    raise FileNotFoundError(
        f"No se encontró {nombre}. Colóquelo junto al notebook o en data/. "
        f"Rutas revisadas:\n- {buscadas}"
    )

RUTA_VIDEOS = localizar_archivo("youtube_videos.csv")
RUTA_COMENTARIOS = localizar_archivo("youtube_comments.csv")

ID_VIDEOS = {"video_id": "string", "channel_id": "string"}
ID_COMENTARIOS = {
    "video_id": "string",
    "comment_id": "string",
    "channel_id": "string",
    "author_channel_id": "string",
}

videos = pd.read_csv(RUTA_VIDEOS, dtype=ID_VIDEOS)
comentarios = pd.read_csv(RUTA_COMENTARIOS, dtype=ID_COMENTARIOS)

print(f"Videos: {videos.shape[0]:,} filas x {videos.shape[1]} columnas")
print(f"Comentarios: {comentarios.shape[0]:,} filas x {comentarios.shape[1]} columnas")
print(f"Archivos: {RUTA_VIDEOS.name}, {RUTA_COMENTARIOS.name}")


Videos: 293 filas x 20 columnas
Comentarios: 406 filas x 17 columnas
Archivos: youtube_videos.csv, youtube_comments.csv


In [ ]:
import re
import json
import unicodedata
from collections import Counter

STOPWORDS_ES = set("""
a al algo algunas algunos ante antes como con contra cual cuales cuando de del desde donde durante
e el ella ellas ellos en entre era eran eres es esa esas ese eso esos esta estaba estaban estado
estas este esto estos fue fueron ha han hasta hay la las le les lo los más me mi mis mucho muy no
nos o otra otro otras otros para pero poco por porque que quien quienes se ser si sin sobre son su
sus te tener tiene todo todos toda todas tu tus un una uno unos unas ya ni qué cómo cuál cuáles
cuándo cuánto cuánta cuántos cuántas dónde quién cuyo cuya cuyos cuyas aquel aquella aquellos
aquellas yo tú él nosotros ustedes usted mío mía tuyos tuyas nuestro nuestra también solo sólo cada
hacer hace hacía hacia menos aquí aqui ahí ahi allí alli pues entonces tan tanto nada nadie nunca siempre todavía
aun aún aunque sino mientras mismo misma mismos mismas vez veces puede pueden podría según tras
mas tambien está esta están estan tiene tienen
""".split())

URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
HASHTAG_RE = re.compile(r"(?<!\w)#([\wáéíóúüñ]+)", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@[\w.-]+", flags=re.IGNORECASE)
CUSTOM_EMOJI_RE = re.compile(r":[a-z0-9_-]+:", flags=re.IGNORECASE)
UNICODE_EMOJI_RE = re.compile(
    "[\U0001F1E6-\U0001FAFF\u2600-\u27BF]",
    flags=re.UNICODE,
)

# NFKC + recorte; conserva NA y no reemplaza IDs por etiquetas
def normalizar_texto_corto(serie):
    def uno(x):
        if pd.isna(x):
            return pd.NA
        return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(x)).strip())
    return serie.astype("string").map(uno)

def extraer_hashtags(texto):
    if pd.isna(texto):
        return []
    return [x.casefold() for x in HASHTAG_RE.findall(str(texto))]

def extraer_emojis(texto):
    if pd.isna(texto):
        return []
    texto = str(texto)
    return CUSTOM_EMOJI_RE.findall(texto) + UNICODE_EMOJI_RE.findall(texto)

# Versión analítica conservadora; el original siempre se guarda aparte
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    s = unicodedata.normalize("NFKC", str(texto)).casefold()
    s = URL_RE.sub(" ", s)
    s = HASHTAG_RE.sub(r" \1 ", s)       # #tema -> tema
    s = MENTION_RE.sub(" ", s)            # elimina la etiqueta visible
    s = CUSTOM_EMOJI_RE.sub(" ", s)
    s = UNICODE_EMOJI_RE.sub(" ", s)
    s = re.sub(r"[^a-záéíóúüñ\s]", " ", s)  # también elimina números
    tokens = [t for t in s.split() if len(t) > 1 and t not in STOPWORDS_ES]
    return " ".join(tokens)

def parsear_conteo_youtube(valor, vacio_como_cero=False):
    """Convierte conteos con separadores y sufijos k/m/mil/millón."""
    if pd.isna(valor) or not str(valor).strip():
        return 0 if vacio_como_cero else pd.NA
    s = unicodedata.normalize("NFKC", str(valor)).casefold().strip()
    s = re.sub(r"vistas?|likes?|me gusta", "", s).strip()
    multiplicador = 1

    if re.search(r"\b(k|mil)\b", s):
        multiplicador = 1_000
        s = re.sub(r"\b(k|mil)\b", "", s)

    elif re.search(r"\b(m|millones?|millón)\b", s):
        multiplicador = 1_000_000
        s = re.sub(r"\b(m|millones?|millón)\b", "", s)
    s = s.replace("\u00a0", "").replace(" ", "")

    if multiplicador == 1:
        s = re.sub(r"[,.]", "", s)

    else:
        s = s.replace(",", ".")

    try:
        return int(round(float(s) * multiplicador))
    except (TypeError, ValueError):
        return pd.NA


## 2.1 Diagnóstico inicial de calidad

El diagnóstico cubre dimensiones, tipos, faltantes, duplicados, constantes, atípicos y
consistencia de las correspondencias entre IDs, nombres y handles.


In [3]:
def perfil(df, nombre):
    return pd.DataFrame({
        "archivo": nombre,
        "variable": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "faltantes": df.isna().sum().values,
        "porcentaje_faltante": (100 * df.isna().mean()).values,
        "unicos_sin_NA": df.nunique(dropna=True).values,
    })

perfil_calidad = pd.concat([
    perfil(videos, "videos"),
    perfil(comentarios, "comentarios")
], ignore_index=True)
perfil_calidad


,archivo,variable,dtype,faltantes,porcentaje_faltante,unicos_sin_NA
0,videos,video_id,string,0,0.000,293
1,videos,title,str,0,0.000,274
2,videos,channel_name,str,0,0.000,97
3,videos,channel_id,string,0,0.000,97
4,videos,source_query,str,0,0.000,21
5,videos,source_group,str,0,0.000,3
6,videos,dataset_sources,str,0,0.000,23
7,videos,channel_handle,str,0,0.000,97
8,videos,published_time,str,13,4.437,80
9,videos,view_count_text,str,13,4.437,259


In [4]:
resumen_duplicados = pd.DataFrame([
    {"archivo": "videos", "filas": len(videos),
     "filas_duplicadas_exactas": int(videos.duplicated().sum()),
     "duplicados_llave": int(videos["video_id"].duplicated().sum())},
    {"archivo": "comentarios", "filas": len(comentarios),
     "filas_duplicadas_exactas": int(comentarios.duplicated().sum()),
     "duplicados_llave": int(comentarios["comment_id"].duplicated().sum())},
])

constantes = []
for nombre, df in [("videos", videos), ("comentarios", comentarios)]:
    for col in df.columns:
        if df[col].nunique(dropna=False) == 1:
            constantes.append({"archivo": nombre, "variable": col,
                                "valor": repr(df[col].dropna().iloc[0]) if df[col].notna().any() else "todo NA"})

print("Duplicados")
print(resumen_duplicados.to_string(index=False))
print("\nVariables constantes")
print(pd.DataFrame(constantes).to_string(index=False))


Duplicados
    archivo  filas  filas_duplicadas_exactas  duplicados_llave
     videos    293                         0                 0
comentarios    406                         0                 0

Variables constantes
    archivo      variable     valor
comentarios     is_pinned np.False_
comentarios viewer_rating   todo NA


In [5]:
# Crear conteos numéricos antes de buscar atípicos.
videos["view_count_from_text"] = videos["view_count_text"].map(parsear_conteo_youtube).astype("Int64")
comentarios["like_count"] = comentarios["like_count_text"].map(
    lambda x: parsear_conteo_youtube(x, vacio_como_cero=True)
).astype("Int64")

def resumen_atipicos(serie, nombre, archivo):
    s = pd.to_numeric(serie, errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "archivo": archivo, "variable": nombre, "min": s.min(), "Q1": q1,
        "mediana": s.median(), "media": s.mean(), "Q3": q3, "max": s.max(),
        "límite_IQR_superior": superior,
        "n_atípicos_IQR": int(((s < inferior) | (s > superior)).sum()),
    }

tabla_atipicos = pd.DataFrame([
    resumen_atipicos(videos["view_count"], "view_count", "videos"),
    resumen_atipicos(comentarios["reply_count"], "reply_count", "comentarios"),
    resumen_atipicos(comentarios["like_count"], "like_count", "comentarios"),
])
tabla_atipicos


,archivo,variable,min,Q1,mediana,media,Q3,max,límite_IQR_superior,n_atípicos_IQR
0,videos,view_count,2,215.000,"1,175.000","60,430.085","7,465.000",8190449,"18,340.000",49
1,comentarios,reply_count,0,0.000,0.000,0.126,0.000,7,0.000,30
2,comentarios,like_count,0,0.000,1.000,5.727,2.000,405,5.000,48


Los valores atípicos se **marcan, no se eliminan**. Son plausibles en métricas de plataformas,
que suelen ser muy asimétricas. Para gráficos se usarán escalas logarítmicas y se reportarán
mediana/cuartiles además de la media.


In [ ]:
def conflictos_mapeo(df, id_col, etiqueta_col, contexto):
    por_id = df.dropna(subset=[id_col]).groupby(id_col)[etiqueta_col].nunique(dropna=True)
    por_etiqueta = df.dropna(subset=[etiqueta_col]).groupby(etiqueta_col)[id_col].nunique(dropna=True)
    
    return {
        "contexto": contexto,
        "id": id_col,
        "etiqueta": etiqueta_col,
        "IDs_con_múltiples_etiquetas": int((por_id > 1).sum()),
        "etiquetas_con_múltiples_IDs": int((por_etiqueta > 1).sum()),
    }

consistencia = pd.DataFrame([
    conflictos_mapeo(videos, "channel_id", "channel_name", "canal en videos"),
    conflictos_mapeo(videos, "channel_id", "channel_handle", "canal en videos"),
    conflictos_mapeo(comentarios, "channel_id", "channel_name", "canal en comentarios"),
    conflictos_mapeo(comentarios, "author_channel_id", "author_name", "autor"),
    conflictos_mapeo(comentarios, "author_channel_id", "author_handle", "autor"),
])

fk_sin_video = int((~comentarios["video_id"].isin(videos["video_id"])).sum())
print(consistencia.to_string(index=False))
print(f"\nComentarios cuyo video_id no existe en videos: {fk_sin_video}")
print(f"channel_handle == owner_handle: {(videos['channel_handle'] == videos['owner_handle']).sum()} de {len(videos)}")
print(f"publish_date == upload_date: {(videos['publish_date'] == videos['upload_date']).sum()} de {len(videos)}")


            contexto                id       etiqueta  IDs_con_múltiples_etiquetas  etiquetas_con_múltiples_IDs
     canal en videos        channel_id   channel_name                            0                            0
     canal en videos        channel_id channel_handle                            0                            0
canal en comentarios        channel_id   channel_name                            0                            0
               autor author_channel_id    author_name                            0                            0
               autor author_channel_id  author_handle                            0                            0

Comentarios cuyo video_id no existe en videos: 0
channel_handle == owner_handle: 293 de 293
publish_date == upload_date: 293 de 293


## 2.2 Variables problemáticas o con precauciones especiales

| Variable | Tratamiento y justificación |
|---|---|
| `viewer_rating` | Excluir de análisis: está completamente vacía. |
| `is_pinned` | Conservar por trazabilidad, pero no modelar: es constante (`False`). |
| `published_time`, `published_text` | Son tiempos relativos al momento de recolección; no convertirlos en fechas exactas sin conocer ese momento. |
| `view_count_text` | Convertir solo como control; usar `view_count` para cálculos porque ya es numérica y las capturas pueden diferir en el tiempo. |
| `like_count_text` | Convertir; el espacio vacío se interpreta como cero mostrado por la interfaz y se documenta. |
| `reply_count` | Usar como atributo agregado. No crear aristas entre usuarios porque faltan autores y texto de las respuestas. |
| `channel_name`, `author_name`, handles | Etiquetas visibles, no identificadores. Pueden cambiar o repetirse fuera de esta muestra. |
| `source_query`, `source_group` | Describen selección/recolección, no el tema definitivo del contenido. |
| `dataset_sources` | Proveniencia; no es una categoría temática. |
| `video_title`, canal en comentarios | Redundantes tras la unión; verificar coherencia y preferir atributos de `videos`. |
| `upload_date` | Coincide con `publish_date` en esta muestra; conservar una como control y evitar doble conteo. |


## 2.3 Normalización de identificadores y nombres

Se aplica normalización Unicode NFKC, recorte de extremos y colapso de espacios internos. No se
cambia mayúsculas/minúsculas en IDs ni se sustituyen IDs por nombres. Las etiquetas originales
continúan disponibles en sus columnas.


In [7]:
columnas_videos = ["video_id", "channel_id", "channel_name", "channel_handle", "owner_handle"]
columnas_comentarios = ["video_id", "comment_id", "channel_id", "author_channel_id",
                        "channel_name", "author_name", "author_handle"]

cambios_normalizacion = []
for archivo, df, columnas in [
    ("videos", videos, columnas_videos),
    ("comentarios", comentarios, columnas_comentarios),
]:
    for col in columnas:
        antes = df[col].astype("string")
        despues = normalizar_texto_corto(df[col])
        cambios = int((antes.fillna("<NA>") != despues.fillna("<NA>")).sum())
        df[col] = despues
        cambios_normalizacion.append({"archivo": archivo, "variable": col, "valores_modificados": cambios})

pd.DataFrame(cambios_normalizacion)


,archivo,variable,valores_modificados
0,videos,video_id,0
1,videos,channel_id,0
2,videos,channel_name,2
3,videos,channel_handle,0
4,videos,owner_handle,0
5,comentarios,video_id,0
6,comentarios,comment_id,0
7,comentarios,channel_id,0
8,comentarios,author_channel_id,0
9,comentarios,channel_name,0


In [8]:
ids_requeridos = {
    "videos.video_id": videos["video_id"],
    "videos.channel_id": videos["channel_id"],
    "comentarios.video_id": comentarios["video_id"],
    "comentarios.comment_id": comentarios["comment_id"],
    "comentarios.channel_id": comentarios["channel_id"],
    "comentarios.author_channel_id": comentarios["author_channel_id"],
}
validacion_ids = pd.DataFrame([
    {"campo": nombre, "faltantes": int(s.isna().sum()), "vacíos": int(s.fillna("").eq("").sum()),
     "únicos": int(s.nunique(dropna=True))}
    for nombre, s in ids_requeridos.items()
])
validacion_ids


,campo,faltantes,vacíos,únicos
0,videos.video_id,0,0,293
1,videos.channel_id,0,0,97
2,comentarios.video_id,0,0,19
3,comentarios.comment_id,0,0,406
4,comentarios.channel_id,0,0,8
5,comentarios.author_channel_id,0,0,332


## 2.4 Conteos de texto a numérico

El parser admite comas o puntos de millares, espacios y abreviaturas `k`, `m`, `mil` y
`millón/millones`. En valores sin abreviatura, comas y puntos se interpretan como separadores de
millares. Los textos no reconocibles se convierten a `NA`. Para `like_count_text`, el valor vacío
se interpreta como cero mostrado, no como dato desconocido.


In [9]:
# Recalcular después de normalizar las columnas de identificación.
videos["view_count_from_text"] = videos["view_count_text"].map(parsear_conteo_youtube).astype("Int64")
comentarios["like_count"] = comentarios["like_count_text"].map(
    lambda x: parsear_conteo_youtube(x, vacio_como_cero=True)
).astype("Int64")
videos["publish_date_dt"] = pd.to_datetime(videos["publish_date"], errors="coerce", utc=True)
videos["upload_date_dt"] = pd.to_datetime(videos["upload_date"], errors="coerce", utc=True)

vista_no_vacia = videos["view_count_text"].fillna("").str.strip().ne("")
likes_vacios = comentarios["like_count_text"].fillna("").str.strip().eq("")
diferencia_vistas = (videos["view_count_from_text"] - videos["view_count"]).abs()

conversion = pd.DataFrame([
    {"variable": "view_count_text", "filas": len(videos),
     "vacíos_origen": int((~vista_no_vacia).sum()),
     "no_válidos_no_vacíos": int((vista_no_vacia & videos["view_count_from_text"].isna()).sum()),
     "resultado_NA": int(videos["view_count_from_text"].isna().sum())},
    {"variable": "like_count_text", "filas": len(comentarios),
     "vacíos_origen": int(likes_vacios.sum()),
     "no_válidos_no_vacíos": int((~likes_vacios & comentarios["like_count"].isna()).sum()),
     "resultado_NA": int(comentarios["like_count"].isna().sum())},
])
print(conversion.to_string(index=False))
print(f"\nview_count_from_text coincide exactamente con view_count en {(diferencia_vistas == 0).sum()} filas;")
print(f"difiere en {(diferencia_vistas > 0).sum()} y no puede compararse en {diferencia_vistas.isna().sum()}.")
print("Las diferencias se conservan como evidencia de capturas en momentos distintos; view_count es la referencia analítica.")


       variable  filas  vacíos_origen  no_válidos_no_vacíos  resultado_NA
view_count_text    293             13                     0            13
like_count_text    406            189                     0             0

view_count_from_text coincide exactamente con view_count en 227 filas;
difiere en 53 y no puede compararse en 13.
Las diferencias se conservan como evidencia de capturas en momentos distintos; view_count es la referencia analítica.


In [10]:
def parsear_lista_json(valor):
    if pd.isna(valor) or not str(valor).strip():
        return []
    salida = json.loads(str(valor))
    if not isinstance(salida, list):
        raise ValueError("Se esperaba una lista JSON")
    return salida

videos["query_hits_list"] = videos["query_hits"].map(parsear_lista_json)
videos["keywords_list"] = videos["keywords"].map(parsear_lista_json)
videos["dataset_sources_list"] = videos["dataset_sources"].map(
    lambda x: [p.strip() for p in str(x).split("|") if p.strip()]
)
comentarios["dataset_sources_list"] = comentarios["dataset_sources"].map(
    lambda x: [p.strip() for p in str(x).split("|") if p.strip()]
)

resumen_listas = pd.DataFrame({
    "variable": ["query_hits", "keywords", "dataset_sources (videos)", "dataset_sources (comentarios)"],
    "listas_vacías": [
        videos["query_hits_list"].map(len).eq(0).sum(),
        videos["keywords_list"].map(len).eq(0).sum(),
        videos["dataset_sources_list"].map(len).eq(0).sum(),
        comentarios["dataset_sources_list"].map(len).eq(0).sum(),
    ],
    "máximo_elementos": [
        videos["query_hits_list"].map(len).max(),
        videos["keywords_list"].map(len).max(),
        videos["dataset_sources_list"].map(len).max(),
        comentarios["dataset_sources_list"].map(len).max(),
    ],
})
resumen_listas


,variable,listas_vacías,máximo_elementos
0,query_hits,0,2
1,keywords,162,53
2,dataset_sources (videos),0,9
3,dataset_sources (comentarios),0,3


In [11]:
videos[["video_id", "view_count_text", "view_count_from_text", "view_count"]].assign(
    diferencia=lambda d: (d["view_count_from_text"] - d["view_count"]).abs()
).sort_values("diferencia", ascending=False).head(10)


,video_id,view_count_text,view_count_from_text,view_count,diferencia
107,Jr1hJ92H_Nc,"59,559 vistas",59559,62941,3382
245,n4xHxJPNr78,"3,152,430 vistas",3152430,3152619,189
180,_1VIxOLyaUQ,"329,964 vistas",329964,330057,93
50,9BIrwWQUdb8,"24,038 vistas",24038,23972,66
0,-5puKGEqcUc,"2,390 vistas",2390,2357,33
127,O6gI0ooBM3E,"16,818 vistas",16818,16786,32
266,tb6LaOWGAIo,"104,891 vistas",104891,104869,22
56,Aa7hIDCRh3g,"504,353 vistas",504353,504374,21
6,0a4_g1R1-aA,"15,436 vistas",15436,15455,19
126,NnKpTEfhWjI,"424,856 vistas",424856,424874,18


## 2.5 y 2.6 Texto original y texto limpio

Decisiones aplicadas a `texto_limpio`:

- minúsculas y normalización Unicode NFKC;
- eliminación de URL;
- hashtags separados: `#Guatemala` pasa a `guatemala`, mientras la lista original se guarda;
- menciones eliminadas del texto analítico, pero los campos de autor/handle siguen intactos;
- eliminación de puntuación y números;
- stopwords españolas incluidas en el notebook, sin descargas externas;
- emojis extraídos a una lista para análisis opcional y eliminados del texto limpio;
- **sin lematización**: requeriría un modelo lingüístico externo y podría introducir errores en
  modismos, nombres propios y texto multilingüe. Esta decisión favorece reproducibilidad. En un
  análisis posterior se puede comparar contra spaCy en español.

El original se conserva para auditoría y para un futuro análisis de sentimiento, donde emojis,
negaciones y puntuación pueden ser informativos.


In [12]:
comentarios["texto_original"] = comentarios["text"].astype("string").fillna("")
comentarios["hashtags"] = comentarios["texto_original"].map(extraer_hashtags)
comentarios["emojis"] = comentarios["texto_original"].map(extraer_emojis)
comentarios["texto_limpio"] = comentarios["texto_original"].map(limpiar_texto)

videos["titulo_original"] = videos["title"].astype("string").fillna("")
videos["descripcion_original"] = videos["description"].astype("string").fillna("")
videos["texto_video_original"] = (
    videos["titulo_original"].str.strip() + " " + videos["descripcion_original"].str.strip()
).str.strip()
videos["hashtags"] = videos["texto_video_original"].map(extraer_hashtags)
videos["emojis"] = videos["texto_video_original"].map(extraer_emojis)
videos["titulo_limpio"] = videos["titulo_original"].map(limpiar_texto)
videos["descripcion_limpia"] = videos["descripcion_original"].map(limpiar_texto)
videos["texto_video_limpio"] = videos["texto_video_original"].map(limpiar_texto)

comentarios[["comment_id", "texto_original", "texto_limpio", "hashtags", "emojis"]].head(8)


,comment_id,texto_original,texto_limpio,hashtags,emojis
0,Ugw-J65a1iYL9hqhELh4AaABAg,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,corrupto amigo vieja fiscal tengo verbose carcel,[],[]
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay policías que le...",jóvenes buscan trabajo tuvieron suerte policías gusta vando ir vestido mujer hubieran ...,[],[]
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es ...",dejaron ganas demandar ilegalidad reuniones virtuales maquila,[],[]
3,Ugw0xgUc2ISpBr5_T654AaABAg,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la sombra.,veremos mafioso mazariegos cárcel buen tiempo sombra,[],[]
4,Ugw1ZzA21njWhaqQQTh4AaABAg,eso es para que salga de USA por su propio pie \nque se auto deporten,salga usa propio pie auto deporten,[],[]
5,Ugw1ZzA21njWhaqQQTh4AaABAg.Aa9Tf29tBouAaB_XZQglTL,Imagine if they had to walk back home.,imagine if they had to walk back home,[],[]
6,Ugw2014OIYf336oytFt4AaABAg,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::hand-purple-b...,buenísima investigacion,[],"[:hand-purple-blue-peace:, :hand-purple-blue-peace:, :hand-purple-blue-peace:]"
7,Ugw2bSVOoNn2pc8ZcUl4AaABAg,"Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están robando.",lleven lonchera sacrifiquense reintevren dinero robando,[],[]


## 2.7 Efecto de la limpieza

No se eliminan filas: comentarios vacíos o duplicados después de limpiar quedan marcados para
revisión. Los resúmenes léxicos posteriores omitirán `texto_limpio == ""`, pero la tabla conserva
los registros y sus IDs.


In [13]:
def efecto_limpieza(original, limpio, entidad):
    original = original.fillna("")
    limpio = limpio.fillna("")
    return {
        "entidad": entidad,
        "registros": len(original),
        "registros_eliminados": 0,
        "modificados": int((original != limpio).sum()),
        "vacíos_antes": int(original.str.strip().eq("").sum()),
        "vacíos_después": int(limpio.str.strip().eq("").sum()),
        "duplicados_antes": int(original.duplicated(keep="first").sum()),
        "duplicados_después": int(limpio.duplicated(keep="first").sum()),
    }

efecto = pd.DataFrame([
    efecto_limpieza(comentarios["texto_original"], comentarios["texto_limpio"], "comentarios"),
    efecto_limpieza(videos["titulo_original"], videos["titulo_limpio"], "títulos"),
    efecto_limpieza(videos["descripcion_original"], videos["descripcion_limpia"], "descripciones"),
])
efecto


,entidad,registros,registros_eliminados,modificados,vacíos_antes,vacíos_después,duplicados_antes,duplicados_después
0,comentarios,406,0,403,0,6,2,11
1,títulos,293,0,293,0,2,19,36
2,descripciones,293,0,267,26,26,58,64


In [14]:
vacios_limpios = comentarios.loc[
    comentarios["texto_limpio"].eq(""),
    ["comment_id", "texto_original", "emojis"]
]
print(f"Comentarios compuestos solo por elementos removidos: {len(vacios_limpios)}")
print(vacios_limpios.to_string(index=False))


Comentarios compuestos solo por elementos removidos: 6
                                       comment_id           texto_original             emojis
                       UgwIlv8gtF3WKYfHSJJ4AaABAg                        😮                [😮]
                       UgwSDn5aIUEZwOlx9c94AaABAg                       ❤🎉             [❤, 🎉]
UgwxjmtBoAozs0ZUAcd4AaABAg.AaAbqoGVGipAaAjNG9oPRN                     que?                 []
UgxEE17R1iQyuDZebVF4AaABAg.A_UFk_pMAbMA_VZ_0Vdu89                       😂😂             [😂, 😂]
UgxEsAK7q3l34wFdZ414AaABAg.A_zCnf5bSajAa0qEPC8tJc ​ @Murmullodelbarrio 👍🇬🇹          [👍, 🇬, 🇹]
                       UgxoLjEjkngSVGZvO1N4AaABAg                   👏👏👏👏👏👏 [👏, 👏, 👏, 👏, 👏, 👏]


In [15]:
# Exportaciones reproducibles opcionales para los siguientes puntos.
SALIDAS = Path.cwd() / "salidas"
SALIDAS.mkdir(exist_ok=True)
videos.to_csv(SALIDAS / "02_youtube_videos_limpio.csv", index=False)
comentarios.to_csv(SALIDAS / "02_youtube_comments_limpio.csv", index=False)
print("Archivos derivados guardados en ./salidas/")


Archivos derivados guardados en ./salidas/


In [16]:
assert videos["video_id"].is_unique
assert comentarios["comment_id"].is_unique
assert comentarios["author_channel_id"].notna().all()
assert comentarios["like_count"].notna().all()
assert len(comentarios) == 406 and len(videos) == 293
print("Validaciones finales superadas; no se perdieron filas ni llaves.")


Validaciones finales superadas; no se perdieron filas ni llaves.


## Conclusión del punto 2

Las llaves son completas y únicas, y no hay duplicados exactos de filas. Los principales riesgos
son la columna totalmente vacía `viewer_rating`, constantes, tiempos relativos, conteos tomados en
momentos distintos y fuerte asimetría. La limpieza mantiene los originales y evita que decisiones
de procesamiento se confundan con información observada.
